# Chatterbox ONNX — 2-sinov: dekoder tezligi

Birinchi sinovda **dekoder** (S3Gen) vaqtning ~85% ini oldi. Bu daftar: (1) isitishdan keyin qayta o'lchaydi, (2) dekoderni int8 ga siqib ko'radi, (3) til modelini tasdiqlaydi.

Runtime → **CPU** → Run all (10–15 daqiqa). Natijani yuboring.

In [ ]:
!pip -q install onnxruntime numpy huggingface_hub
import time, os, numpy as np, onnxruntime as ort
from huggingface_hub import hf_hub_download, list_repo_files

REPO = "onnx-community/chatterbox-multilingual-ONNX"
LOCAL = "/content/cb_onnx"
CHOSEN = {"speech_encoder": "onnx/speech_encoder.onnx", "embed_tokens": "onnx/embed_tokens.onnx",
          "language_model": "onnx/language_model_q4.onnx", "conditional_decoder": "onnx/conditional_decoder.onnx"}
files = list_repo_files(REPO)
paths = {}
for n, f in CHOSEN.items():
    paths[n] = hf_hub_download(REPO, f, local_dir=LOCAL)
    if f + "_data" in files:
        hf_hub_download(REPO, f + "_data", local_dir=LOCAL)
print({n: os.path.getsize(p) for n, p in paths.items()})

def load(name, threads, path=None):
    so = ort.SessionOptions()
    so.intra_op_num_threads = threads
    so.inter_op_num_threads = 1
    return ort.InferenceSession(path or paths[name], so, providers=["CPUExecutionProvider"])

def run(sess, feeds):
    names = {i.name for i in sess.get_inputs()}
    return sess.run(None, {k: v for k, v in feeds.items() if k in names})

print("CPU yadrolari:", os.cpu_count())


In [ ]:
# 1) Dekoder (S3Gen) — asosiy to'siq. Isitish (warm-up) + qayta o'lchash, token soniga bog'liqlik
se = load("speech_encoder", 2)
audio = (np.random.randn(1, 24000 * 6) * 0.05).astype(np.float32)
cond_emb, prompt_token, ref_x, prompt_feat = run(se, {"audio_values": audio})[:4]
del se

def dec_time(sess, n_tokens):
    toks = np.random.randint(0, 6561, (1, n_tokens)).astype(np.int64)
    speech = np.concatenate([prompt_token, toks], 1)
    t = time.time()
    run(sess, {"speech_tokens": speech, "speaker_embeddings": ref_x, "speaker_features": prompt_feat})
    return time.time() - t

for th in (1, 2):
    cd = load("conditional_decoder", th)
    dec_time(cd, 30)  # warm-up
    for n in (50, 100):
        t = dec_time(cd, n)
        print(f"[fp32 dekoder, {th} oqim] {n} token = {n/25:.1f} s audio -> {t:.1f} s (RTF {t/(n/25):.1f})")
    del cd


In [ ]:
# 2) Dekoderni int8 ga siqib ko'ramiz (dinamik kvantlash). Xato bo'lsa — shu haqda yozadi.
from onnxruntime.quantization import quantize_dynamic, QuantType
src = paths["conditional_decoder"]
dst = os.path.join(LOCAL, "onnx", "conditional_decoder_int8.onnx")
try:
    t = time.time()
    quantize_dynamic(src, dst, weight_type=QuantType.QInt8, use_external_data_format=True)
    print(f"kvantlash {time.time()-t:.0f} s, hajm:", sum(os.path.getsize(os.path.join(LOCAL,'onnx',f)) for f in os.listdir(os.path.join(LOCAL,'onnx')) if f.startswith('conditional_decoder_int8'))/1e6, "MB")
    for th in (1, 2):
        cdq = load("conditional_decoder", th, dst)
        dec_time(cdq, 30)
        for n in (50, 100):
            t = dec_time(cdq, n)
            print(f"[int8 dekoder, {th} oqim] {n} token = {n/25:.1f} s audio -> {t:.1f} s (RTF {t/(n/25):.1f})")
        del cdq
except Exception as e:
    print("KVANTLASH XATOSI:", type(e).__name__, str(e)[:500])


In [ ]:
# 3) Til modeli (matn -> ovoz tokenlari) — qayta tasdiqlash
et = load("embed_tokens", 2); lm = load("language_model", 2)
exg = np.array([0.5], np.float32)
ids = np.random.randint(10, 500, (1, 60)).astype(np.int64)
pos = (np.arange(60)[None, :] - 1).astype(np.int64)
emb = run(et, {"input_ids": ids, "position_ids": pos, "exaggeration": exg})[0]
x = np.concatenate([cond_emb, emb], 1)
pin = [i for i in lm.get_inputs() if i.name.startswith("past_key_values")]
H = pin[0].shape[1] if isinstance(pin[0].shape[1], int) else 16
D = pin[0].shape[3] if isinstance(pin[0].shape[3], int) else 64
names = [i.name for i in pin]
past = {n: np.zeros((1, H, 0, D), np.float32) for n in names}
mask = np.ones((1, x.shape[1]), np.int64)
t = time.time(); out = run(lm, dict(inputs_embeds=x, attention_mask=mask, **past)); tp = time.time() - t
past = dict(zip(names, out[1:])); steps = []
for i in range(100):
    tok = np.random.randint(0, 6561, (1, 1)).astype(np.int64)
    t = time.time()
    e = run(et, {"input_ids": tok, "position_ids": np.array([[i + 1]], np.int64), "exaggeration": exg})[0]
    mask = np.concatenate([mask, np.ones((1, 1), np.int64)], 1)
    out = run(lm, dict(inputs_embeds=e, attention_mask=mask, **past)); past = dict(zip(names, out[1:]))
    steps.append(time.time() - t)
tl = tp + sum(steps)
print(f"[til modeli] prefill {tp:.2f} s, token boshiga {1000*np.mean(steps[10:]):.0f} ms; 4 s audio uchun jami {tl:.1f} s (RTF {tl/4:.2f})")
